# Allocate a Hubbard phase-estimation error budget

Estimate the T cost for phase estimation with a second-order Hubbard product
formula. Compare plain rotations, Hamming-weight phasing and a stripped
Suzuki schedule. These are resource estimates for phase estimation.

Adapted from the [Qualtran v0.7.0 tutorial](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/bloqs/chemistry/trotter/hubbard/qpe_cost_optimization.ipynb), Copyright 2023 Google LLC, Apache-2.0.

## Three error contributions

For order $p$ and spectral-norm bound $\xi$, the Suzuki error budget
$\Delta_{TS}$ permits $t=(\Delta_{TS}/\xi)^{1/p}$. The circuit synthesis
budget $\Delta_{HT}$ sets the precision $\epsilon_R=\Delta_{HT}t/N_R$
for $N_R$ rotations. The adaptive phase-estimation estimate uses
$N_{PE}=0.76\pi/(\Delta_{PE}t)$ steps. Allocate the total error among
those three terms and multiply the resulting step count by the sum of
direct T equivalents and synthesized rotation costs.

In [ ]:
from io import BytesIO
from math import ceil, isclose, pi

from IPython.display import Image, SVG, display
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize
import sympy as sp

from flagquantum.algorithms.primitives.hubbard_trotter import source_hubbard_suzuki_cost
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.compiler._component_trace.terminal_t_cost import rotation_t_cost
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg


def timestep(delta_ts, xi, order):
    return (delta_ts / xi) ** (1 / order)


def qpe_t_count(delta_pe, delta_ts, delta_ht, rotations, direct_t, xi, order):
    dt = timestep(delta_ts, xi, order)
    synthesis = rotations * rotation_t_cost(delta_ht * dt / rotations) if rotations else 0
    return (0.76 * pi / (delta_pe * dt)) * (direct_t + synthesis)


def gate_counts(model):
    gates = estimate_component_resources(model).gates
    return gates.total_t_count(ts_per_rotation=0), gates.rotation


def show_figure(figure):
    with BytesIO() as buffer:
        figure.savefig(buffer, format="png")
        plt.close(figure)
        display(Image(data=buffer.getvalue()))

## Eight-by-eight example

Use the paper's spectral-norm bound 530 and extensive energy error
0.0051 times the number of sites. Start by assigning 50% to Suzuki,
45% to phase estimation and 5% to synthesis. The resource graph
shows the declared Suzuki descendants.

In [ ]:
length, hubb_u, xi, order = 8, 4.0, 530.0, 2
epsilon = 0.0051 * length**2
initial_ts, initial_pe, initial_ht = (fraction * epsilon for fraction in (0.50, 0.45, 0.05))
dt = timestep(initial_ts, xi, order)
plain = source_hubbard_suzuki_cost(length, hubb_u, dt)
direct_t, rotations = gate_counts(plain)
cost = qpe_t_count(initial_pe, initial_ts, initial_ht, rotations, direct_t, xi, order)
assert isclose(initial_ts + initial_pe + initial_ht, epsilon)
assert (direct_t, rotations) == (768, 320)
assert f"{cost:.3e}" == "1.049e+07"
print("Timestep:", dt)
print("Single-rotation precision:", initial_ht * dt / rotations)
print(f"Initial T estimate: {cost:.3e}")
display(SVG(data=render_call_graph_svg(export_modeled_call_graph_dot(plain)).decode()))

## Explore the error surface

Fix the synthesis allocation at 0.00001. The surface and contour
show how changing the phase-estimation and Suzuki allowances affects cost.
Contours of total allocation identify budget boundaries.

In [ ]:
pe, ts = np.meshgrid(np.logspace(-2, -0.5, 20), np.logspace(-2, -0.5, 20))
fixed_ht = 1e-5
costs = np.asarray([
    qpe_t_count(float(x), float(y), fixed_ht, rotations, direct_t, xi, order)
    for x, y in zip(pe.flat, ts.flat, strict=True)
]).reshape(pe.shape)
log_costs = np.log10(costs)

fig, axis = plt.subplots(figsize=(6.8, 4.5), subplot_kw={"projection": "3d"}, layout="constrained")
axis.plot_surface(pe, ts, log_costs, cmap="coolwarm", linewidth=0, antialiased=False)
axis.set(xlabel="Phase-estimation error", ylabel="Suzuki error", zlabel="log₁₀ T estimate")
show_figure(fig)

fig, axis = plt.subplots(figsize=(6.5, 4.5), layout="constrained")
contours = axis.contour(pe, ts, log_costs, 25, cmap="coolwarm")
axis.clabel(axis.contour(pe, ts, pe + ts + fixed_ht, 10, colors="0.45"), fontsize=8)
axis.contour(pe, ts, pe + ts + fixed_ht, levels=[epsilon], linestyles="dotted")
fig.colorbar(contours, ax=axis, label="log₁₀ T estimate")
axis.set(xlabel="Phase-estimation error", ylabel="Suzuki error", title=f"Target total error {epsilon:.4f}")
show_figure(fig)

## Sweep a fixed total budget

For each synthesis fraction, the remaining budget is split between
Suzuki and phase estimation. Phase-estimation cost grows as its
allocation approaches zero.

In [ ]:
fig, axis = plt.subplots(figsize=(6.5, 4.5), layout="constrained")
for fraction in (1e-4, 1e-3, 1e-2, 1e-1):
    ht = fraction * epsilon
    allocations = np.linspace(0.05 * epsilon, epsilon - ht, 100)[:-1]
    values = [
        qpe_t_count(epsilon - value - ht, value, ht, rotations, direct_t, xi, order)
        for value in allocations
    ]
    axis.plot(allocations / epsilon, values, label=f"Synthesis fraction {fraction:g}")
axis.set(xlabel="Suzuki fraction", ylabel="Estimated T count", yscale="log")
axis.legend()
show_figure(fig)

## Optimize the error split

The locked tutorial scans 100 synthesis allocations and solves a bounded
one-variable problem for the Suzuki allocation at each point. SciPy's
objective receives a one-element array. Extract its scalar before
calling the scalar rotation-synthesis formula. This preserves the
original numerical objective and its rounded per-rotation cost.

In [ ]:
def optimize_budget(rotations, direct_t):
    best = None
    for ht in np.linspace(0.005 * epsilon, 0.1 * epsilon, 100):
        ht = float(ht)

        def objective(candidate):
            ts = np.asarray(candidate).item()
            return qpe_t_count(epsilon - ts - ht, ts, ht, rotations, direct_t, xi, order)

        available = epsilon - ht
        solution = minimize(
            objective, x0=0.7 * available,
            bounds=[(0.001 * epsilon, 0.9 * available)],
        )
        ts = solution.x.item()
        row = ht, ts, available - ts, objective(ts)
        if best is None or row[-1] < best[-1]:
            best = row
    return best


ordinary = optimize_budget(rotations, direct_t)
assert isclose(sum(ordinary[:3]), epsilon)
assert isclose(ordinary[-1], 9.226e6, rel_tol=0.005)
print("Best sampled split (synthesis, Suzuki, phase estimation):", ordinary[:3])
print(f"Ordinary T estimate: {ordinary[-1]:.3e}")

## Hamming-weight phasing and layer stripping

Hamming-weight phasing trades rotation synthesis for count and erase work.
The stripped Suzuki step combines adjacent interaction layers when steps
are repeated. Use the same objective and budget search for both schedules.
The paper comparison uses a distinct Toffoli costing convention.

In [ ]:
assertions = ((False, (2008, 60), 3.071e6), (True, (1760, 48), 2.565e6))
for stripped, expected_counts, expected_cost in assertions:
    model = source_hubbard_suzuki_cost(
        length, hubb_u, dt, hamming_weight=True, strip_layer=stripped,
    )
    counts = gate_counts(model)
    assert counts == expected_counts
    best = optimize_budget(counts[1], counts[0])
    assert isclose(sum(best[:3]), epsilon)
    assert isclose(best[-1], expected_cost, rel_tol=0.005)
    label = "Stripped HWP" if stripped else "Complete HWP"
    print(f"{label}: {counts[0]} direct T, {counts[1]} rotations, {best[-1]:.3e} estimated T")
print(f"Paper comparison: {1.7e6 + 4 * 1.8e5:.3e}")

## Symbolic estimate

For a generic angle, an ordinary complete step contributes
$12L^2$ direct T equivalents and $5L^2$ rotations. The latter counts
three hopping layers and two interaction layers. Substitute these
counts and the three error allocations into the symbolic QPE estimate.
Check it both for the initial budget and for the visualization's fixed
synthesis budget.

In [ ]:
L, p = sp.symbols("L p", positive=True, integer=True)
d_pe, d_ts, d_ht, s_xi = sp.symbols("Delta_PE Delta_TS Delta_HT xi", positive=True)
s_direct, s_rotations = 12 * L**2, 5 * L**2
assert (s_direct.subs(L, length), s_rotations.subs(L, length)) == (direct_t, rotations)
s_dt = timestep(d_ts, s_xi, p)
s_precision = d_ht * s_dt / s_rotations
estimate = (sp.Rational(76, 100) * sp.pi / (d_pe * s_dt)) * (
    s_direct + s_rotations * rotation_t_cost(s_precision)
)
for ht in (initial_ht, fixed_ht):
    symbolic = estimate.subs({
        L: length, p: order, s_xi: xi, d_pe: initial_pe, d_ts: initial_ts, d_ht: ht,
    }).evalf()
    numeric = qpe_t_count(initial_pe, initial_ts, ht, rotations, direct_t, xi, order)
    assert ceil(symbolic) == ceil(numeric)
display(estimate)